# Notebook 01 — Neo4j Knowledge Graph: Data Ingestion
**Layer:** Foundation — no LLM dependency  
**Inputs:** `02_feature_layer/training/outputs/hdb_feature_table_20260403.csv`  
**Outputs:** 260,699 `:Flat` nodes · 9,710 `:Block` nodes · 26 `:Town` nodes · `NEAR_*` relationships

## 1.1 Imports and paths

In [1]:
import pandas as pd
from pathlib import Path
from neo4j import GraphDatabase
from tqdm import tqdm

ROOT = Path('D:\Master Degree\Projects\Transparent_AI\PropertyLens')
FEATURE_DIR = ROOT / '02_feature_layer' / 'training' / 'outputs'
#FEATURE_DIR = Path("02_feature_layer/training/outputs")
CSV_PATH    = FEATURE_DIR / "hdb_feature_table_20260412.csv"

NEO4J_URI  = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "pass@Word123"   # <-- update

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
print("Neo4j driver ready.")

Neo4j driver ready.


## 1.2 Load CSV and decode one-hot columns

In [5]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols")

TOWN_COLS  = [c for c in df.columns if c.startswith("town_")]
TYPE_COLS  = [c for c in df.columns if c.startswith("flat_type_")]
MODEL_COLS = [c for c in df.columns if c.startswith("flat_model_")]

def decode_onehot(row, cols, prefix):
    for c in cols:
        if row[c] == 1:
            return c.replace(prefix, "")
    return "UNKNOWN"

df["town"]       = df.apply(lambda r: decode_onehot(r, TOWN_COLS,  "town_"),       axis=1)
df["flat_type"]  = df.apply(lambda r: decode_onehot(r, TYPE_COLS,  "flat_type_"),  axis=1)
df["flat_model"] = df.apply(lambda r: decode_onehot(r, MODEL_COLS, "flat_model_"), axis=1)
print("Decoded. Sample towns:", df["town"].unique()[:5].tolist())

Loaded: 260,699 rows x 73 cols
Decoded. Sample towns: ['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH']


## 1.3 Create Neo4j constraints

In [6]:
CONSTRAINTS = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (f:Flat)  REQUIRE f.address_key IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (b:Block) REQUIRE b.block_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (t:Town)  REQUIRE t.name IS UNIQUE",
]
with driver.session() as session:
    for cql in CONSTRAINTS:
        session.run(cql)
print("Constraints created.")

Constraints created.


## 1.4 Batch ingest CSV → Neo4j nodes

In [7]:
CHUNK_SIZE = 5000
total = 0

with driver.session() as session:
    for chunk in tqdm(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE), desc="Ingesting"):
        chunk["town"]       = chunk.apply(lambda r: decode_onehot(r, TOWN_COLS,  "town_"),       axis=1)
        chunk["flat_type"]  = chunk.apply(lambda r: decode_onehot(r, TYPE_COLS,  "flat_type_"),  axis=1)
        chunk["flat_model"] = chunk.apply(lambda r: decode_onehot(r, MODEL_COLS, "flat_model_"), axis=1)

        records = []
        for _, row in chunk.iterrows():
            records.append({
                "address_key": str(row["address_key"]),
                "block_id":    str(row["address_key"]).split("_")[0],
                "town":        row["town"],
                "props": {
                    "resale_price":                          float(row["resale_price"]),
                    "transaction_year":                      int(row["transaction_year"]),
                    "level_mid":                             float(row["level_mid"]),
                    "lease_remaining_years":                 float(row["lease_remaining_years"]),
                    "floor_area_sqm":                        float(row["floor_area_sqm"]),
                    "room_count":                            float(row["room_count"]),
                    "dist_to_mrt_m":                         float(row["dist_to_mrt_m"]),
                    "orientation_score":                     float(row["orientation_score"]),
                    "dist_to_highway_m":                     float(row["dist_to_highway_m"]),
                    "dist_to_foodcourt_m":                   float(row["dist_to_foodcourt_m"]),
                    "dist_to_nearest_mall_m":                float(row["dist_to_nearest_mall_m"]),
                    "mall_count_3km":                        float(row["mall_count_3km"]),
                    "mall_weighted_access_3km":              float(row["mall_weighted_access_3km"]),
                    "dist_to_nearest_school_m":              float(row["dist_to_nearest_school_m"]),
                    "school_count_1km":                      float(row["school_count_1km"]),
                    "primary_school_quality_1km_weighted":   float(row["primary_school_quality_1km_weighted"]),
                    "primary_school_top_quality_1km":        float(row["primary_school_top_quality_1km"]),
                    "primary_school_count_1km":              float(row["primary_school_count_1km"]),
                    "flat_type":  row["flat_type"],
                    "flat_model": row["flat_model"],
                    "town":       row["town"],
                }
            })

        session.run(
            "UNWIND $records AS r "
            "MERGE (f:Flat {address_key: r.address_key}) SET f += r.props "
            "MERGE (b:Block {block_id: r.block_id}) "
            "MERGE (t:Town {name: r.town}) "
            "MERGE (f)-[:IN_BLOCK]->(b) "
            "MERGE (b)-[:IN_TOWN]->(t)",
            records=records
        )
        total += len(chunk)

print(f"Ingested {total:,} rows.")

Ingesting: 53it [01:29,  1.69s/it]

Ingested 260,699 rows.


## 1.5 Create NEAR_* relationships

In [8]:
DIST_COLS = [
    "address_key","dist_to_mrt_m","dist_to_foodcourt_m",
    "dist_to_nearest_mall_m","mall_count_3km","mall_weighted_access_3km",
    "dist_to_nearest_school_m","school_count_1km",
    "primary_school_quality_1km_weighted",
]
REL_Q = (
    "UNWIND $records AS r MATCH (f:Flat {address_key: r.address_key}) "
    "MERGE (f)-[:NEAR_MRT    {distance_m: r.dist_to_mrt_m}]->(f) "
    "MERGE (f)-[:NEAR_HAWKER {distance_m: r.dist_to_foodcourt_m}]->(f) "
    "MERGE (f)-[:NEAR_MALL   {distance_m: r.dist_to_nearest_mall_m, "
    "                          count_3km: r.mall_count_3km, "
    "                          weighted_access: r.mall_weighted_access_3km}]->(f) "
    "MERGE (f)-[:NEAR_SCHOOL {distance_m: r.dist_to_nearest_school_m, "
    "                          count_1km: r.school_count_1km, "
    "                          quality_weighted: r.primary_school_quality_1km_weighted}]->(f)"
)
with driver.session() as session:
    for chunk in tqdm(pd.read_csv(CSV_PATH, usecols=DIST_COLS, chunksize=5000), desc="NEAR_* rels"):
        session.run(REL_Q, records=chunk.fillna(0).to_dict("records"))
print("Relationships created.")

NEAR_* rels: 53it [00:22,  2.37it/s]

Relationships created.


## 1.6 Validation

In [9]:
checks = {
    "Flat nodes":    "MATCH (f:Flat)  RETURN count(f) AS n",
    "Block nodes":   "MATCH (b:Block) RETURN count(b) AS n",
    "Town nodes":    "MATCH (t:Town)  RETURN count(t) AS n",
    "IN_BLOCK rels": "MATCH ()-[:IN_BLOCK]->() RETURN count(*) AS n",
    "NEAR_MRT rels": "MATCH ()-[:NEAR_MRT]->()  RETURN count(*) AS n",
}
with driver.session() as session:
    for label, q in checks.items():
        n = session.run(q).single()["n"]
        print(f"  {label:<20} {n:>10,}")
driver.close()
print("Notebook 01 complete.")

  Flat nodes                9,710
  Block nodes               9,710
  Town nodes                   26
  IN_BLOCK rels             9,710
  NEAR_MRT rels             9,710
Notebook 01 complete.
